In [0]:
"""
06_factory_dashboard.py

Executive Manufacturing Dashboard

Sources:
    machine_kpis
    quality_kpis
    production_kpis
    material_kpis
    packaging_kpis

Target:
    factory_dashboard

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    current_timestamp,
    sum,
)

# ============================================================
# Executive Factory Dashboard
# ============================================================

@dlt.table(
    name="factory_dashboard",
    comment="Executive Manufacturing Dashboard.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def factory_dashboard():

    machine = dlt.read("machine_kpis")

    quality = dlt.read("quality_kpis")

    production = dlt.read("production_kpis")

    material = dlt.read("material_kpis")

    packaging = dlt.read("packaging_kpis")

    machine_summary = (

        machine

        .agg(

            sum(
                "operations_completed"
            ).alias(
                "operations_completed"
            ),

            avg(
                "average_cycle_time_sec"
            ).alias(
                "average_cycle_time_sec"
            ),

            avg(
                "average_force_kn"
            ).alias(
                "average_force_kn"
            ),

            avg(
                "pass_rate"
            ).alias(
                "machine_pass_rate"
            ),

        )

    )

    quality_summary = (

        quality

        .agg(

            sum(
                "tests_completed"
            ).alias(
                "tests_completed"
            ),

            sum(
                "passed_tests"
            ).alias(
                "passed_tests"
            ),

            sum(
                "failed_tests"
            ).alias(
                "failed_tests"
            ),

            avg(
                "pass_rate"
            ).alias(
                "quality_pass_rate"
            ),

        )

    )

    production_summary = (

        production

        .agg(

            sum(
                "work_orders_created"
            ).alias(
                "work_orders_created"
            ),

            sum(
                "executions_started"
            ).alias(
                "executions_started"
            ),

            sum(
                "products_started"
            ).alias(
                "products_started"
            ),

            sum(
                "sap_orders"
            ).alias(
                "sap_orders"
            ),

        )

    )

    material_summary = (

        material

        .agg(

            sum(
                "materials_scanned"
            ).alias(
                "materials_scanned"
            ),

            sum(
                "successful_scans"
            ).alias(
                "successful_scans"
            ),

            sum(
                "failed_scans"
            ).alias(
                "failed_scans"
            ),

            avg(
                "scan_success_rate"
            ).alias(
                "material_scan_success_rate"
            ),

        )

    )

    packaging_summary = (

        packaging

        .agg(

            sum(
                "packages_completed"
            ).alias(
                "packages_completed"
            ),

            sum(
                "ready_for_shipment"
            ).alias(
                "ready_for_shipment"
            ),

            sum(
                "not_ready_for_shipment"
            ).alias(
                "not_ready_for_shipment"
            ),

            avg(
                "shipment_readiness_rate"
            ).alias(
                "shipment_readiness_rate"
            ),

        )

    )

    return (

        machine_summary

        .crossJoin(
            quality_summary
        )

        .crossJoin(
            production_summary
        )

        .crossJoin(
            material_summary
        )

        .crossJoin(
            packaging_summary
        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )